# Análise de Clientes: LTV, RFV e Perfil de Compra

**Objetivo de Negócio:** Identificar nossos melhores clientes, calcular o valor que geram ao longo do tempo (LTV) e entender a distribuição da receita. Isso ajudará a diretoria a focar seus esforços comerciais e entender o comportamento da base.

In [5]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# Conectar ao DuckDB e ler a dimensão de clientes processada
con = duckdb.connect()
dim_clientes_path = r"E:\repo\lh_nautical_analise\data\processed\dim_clientes.parquet"
df_clientes = con.execute(f"SELECT * FROM read_parquet('{dim_clientes_path}')").df()

print(f"Total de clientes na base: {len(df_clientes):,}")


Total de clientes na base: 3,998


## 1. Cálculo do RFV (Recência, Frequência, Valor)

Vamos estipular a data atual (corte) como `2026-08-10` para calcular há quantos dias o cliente não compra (Recência).

In [6]:
# Definir a data de corte do projeto
data_corte = pd.to_datetime('2026-08-10')
df_clientes['data_ultima_compra'] = pd.to_datetime(df_clientes['data_ultima_compra'])

# Calcular Recência em dias
df_clientes['recencia_dias'] = (data_corte - df_clientes['data_ultima_compra']).dt.days

# Preencher clientes sem compras (nulos)
df_clientes['recencia_dias'] = df_clientes['recencia_dias'].fillna(9999)

# Filtrar apenas quem já comprou pelo menos uma vez
df_ativos = df_clientes[df_clientes['total_pedidos'] > 0].copy()

print(f"Clientes ativos (compras > 0): {len(df_ativos):,}")
df_ativos[['id', 'recencia_dias', 'total_pedidos', 'receita_total', 'total_descontos']].head()

Clientes ativos (compras > 0): 3,998


,id,recencia_dias,total_pedidos,receita_total,total_descontos
0,1,137,18,565852.77,9097.00
1,2,149,31,814206.81,9322.09
2,3,117,26,641848.80,8542.19
3,4,47,24,723364.72,7275.45
4,5,122,25,727112.21,23801.82


## 2. A Regra de Pareto (Concentração de Receita)

O Princípio de Pareto sugere que uma pequena parcela de clientes gera a maior parte da receita (Regra 80/20). Vamos verificar como é essa distribuição real na LH Nautical.

In [7]:
df_ativos = df_ativos.sort_values(by='receita_total', ascending=False)
df_ativos['receita_acumulada'] = df_ativos['receita_total'].cumsum()
df_ativos['perc_receita_acumulada'] = df_ativos['receita_acumulada'] / df_ativos['receita_total'].sum()
df_ativos['rank'] = np.arange(1, len(df_ativos) + 1)
df_ativos['perc_clientes'] = df_ativos['rank'] / len(df_ativos)

# Encontrar o ponto de 80% da receita
top_clientes_80 = df_ativos[df_ativos['perc_receita_acumulada'] <= 0.8]
perc_top_clientes = len(top_clientes_80) / len(df_ativos)

print(f"Fato Observado: {perc_top_clientes:.1%} dos clientes geram 80% de toda a receita da LH Nautical.")

Fato Observado: 71.6% dos clientes geram 80% de toda a receita da LH Nautical.


## 3. Segmentação RFV (Adaptação para Varejo Pulverizado)

Como vimos que não existem "baleias" (grandes contas B2B monopolizando a receita), nossa segmentação não focará em tickets extremos, mas sim no **engajamento e retenção de uma base B2C**.

- **Campeões (Champions):** Compram muito, frequentemente, e recentemente.
- **Fiéis Recentes:** Frequência moderada, compraram há pouco tempo.
- **Em Risco / Hibernando:** Compraram bem no passado, mas estão sumindo (alta recência).
- **Compradores Únicos (One-timers):** Compraram uma única vez.

In [8]:
def classificar_cliente_b2c(row):
    if row['total_pedidos'] >= 4 and row['recencia_dias'] <= 90:
        return 'Campeões'
    elif row['total_pedidos'] >= 2 and row['recencia_dias'] <= 180:
        return 'Fiéis Recentes'
    elif row['total_pedidos'] >= 2 and row['recencia_dias'] > 180:
        return 'Em Risco / Hibernando'
    elif row['total_pedidos'] == 1 and row['recencia_dias'] <= 90:
        return 'Novos (Única Compra)'
    else:
        return 'Esporádicos Antigos'

df_ativos['Segmento'] = df_ativos.apply(classificar_cliente_b2c, axis=1)
resumo_segmento = df_ativos.groupby('Segmento').agg({
    'id': 'count',
    'receita_total': 'sum',
    'total_pedidos': 'mean',
    'recencia_dias': 'mean'
}).reset_index().rename(columns={'id': 'Qtd Clientes', 'receita_total': 'Receita Total', 'total_pedidos': 'Frequência Média', 'recencia_dias': 'Recência Média'})

resumo_segmento['% Receita'] = resumo_segmento['Receita Total'] / resumo_segmento['Receita Total'].sum()
resumo_segmento.sort_values(by='% Receita', ascending=False)

,Segmento,Qtd Clientes,Receita Total,Frequência Média,Recência Média,% Receita
0,Campeões,1874,1.231832e+09,22.899680,45.142476,0.480351
2,Fiéis Recentes,1652,1.058829e+09,22.368039,129.507869,0.412889
1,Em Risco / Hibernando,472,2.737808e+08,20.192797,241.542373,0.106760


## 4. Insight de Negócio (Framework F-H-R)

**Fato Observado:** A receita da LH Nautical é extremamente pulverizada e distribuída. Precisamos de 71,6% da base de clientes para formar 80% do faturamento da companhia, indicando completa ausência da regra clássica de Pareto (20/80).

**Hipótese:** O perfil da empresa é massivamente de varejo B2C (consumidor final), não existindo contas corporativas gigantes (B2B/Atacadistas) que sustentem a receita sozinhas.

**Recomendação:** 
1. **Marketing de Escala:** Não vale a pena ter Key Account Management (gerentes dedicados a contas VIPs). Os esforços devem focar em escalar comunicações automatizadas (CRM, e-mail marketing, réguas de relacionamento).
2. **Sistema de Recomendação:** Como a base é pulverizada, a melhor alavanca de vendas é construir um sistema de recomendação personalizado para o e-commerce aumentar a taxa de recompra e o cross-sell para essa massa de consumidores. 